In [2]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

from pybaseball import pitching_stats


In [5]:
all_years_pitching = []
for year in range(2019, 2025):
    df = pitching_stats(year, qual=20)  # min 20 IP
    df['Season'] = year
    all_years_pitching.append(df)

statcast = pd.concat(all_years_pitching, ignore_index=True)
print(f"Total rows: {len(pitching)}")
print(statcast.columns.tolist())

Total rows: 3072
['IDfg', 'Season', 'Name', 'Team', 'Age', 'W', 'L', 'WAR', 'ERA', 'G', 'GS', 'CG', 'ShO', 'SV', 'BS', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'BB', 'IBB', 'HBP', 'WP', 'BK', 'SO', 'GB', 'FB', 'LD', 'IFFB', 'Balls', 'Strikes', 'Pitches', 'RS', 'IFH', 'BU', 'BUH', 'K/9', 'BB/9', 'K/BB', 'H/9', 'HR/9', 'AVG', 'WHIP', 'BABIP', 'LOB%', 'FIP', 'GB/FB', 'LD%', 'GB%', 'FB%', 'IFFB%', 'HR/FB', 'IFH%', 'BUH%', 'Starting', 'Start-IP', 'Relieving', 'Relief-IP', 'RAR', 'Dollars', 'tERA', 'xFIP', 'WPA', '-WPA', '+WPA', 'RE24', 'REW', 'pLI', 'inLI', 'gmLI', 'exLI', 'Pulls', 'WPA/LI', 'Clutch', 'FB% 2', 'FBv', 'SL%', 'SLv', 'CT%', 'CTv', 'CB%', 'CBv', 'CH%', 'CHv', 'SF%', 'SFv', 'KN%', 'KNv', 'XX%', 'PO%', 'wFB', 'wSL', 'wCT', 'wCB', 'wCH', 'wSF', 'wKN', 'wFB/C', 'wSL/C', 'wCT/C', 'wCB/C', 'wCH/C', 'wSF/C', 'wKN/C', 'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%', 'Zone%', 'F-Strike%', 'SwStr%', 'HLD', 'SD', 'MD', 'ERA-', 'FIP-', 'xFIP-', 'K%', 'BB%', 'SIERA', 'RS/

In [ ]:
def run_regression_analysis(target, features, data):
    print(f"\n{'='*60}")
    print(f"  TARGET VARIABLE: {target}")
    print(f"{'='*60}")

    cols = features + [target]
    corr = data[cols].corr()[target]
    selected_features = corr[(corr.abs() > 0.5) & (corr.index != target)].index.tolist()
    print("Selected features:", selected_features)

    results = {'target': target, 'selected_features': selected_features, 'models': {}}

    if len(selected_features) == 0:
        print(f"No features with |correlation| > 0.5 found for {target}. Skipping.")
        return results

    X = data[selected_features].dropna()
    y = data.loc[X.index, target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_scaled = scaler.fit_transform(X)

    # --- Linear Regression ---
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    lr_pred = lr_model.predict(X_test_scaled)
    lr_cv = cross_val_score(lr_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Linear Regression'] = {
        'r2': r2_score(y_test, lr_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, lr_pred)),
        'cv_r2_mean': lr_cv.mean(),
        'cv_r2_std': lr_cv.std()
    }

    # --- Ridge Regression ---
    ridge_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_pred = ridge_model.predict(X_test_scaled)
    ridge_cv = cross_val_score(ridge_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Ridge Regression'] = {
        'r2': r2_score(y_test, ridge_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, ridge_pred)),
        'cv_r2_mean': ridge_cv.mean(),
        'cv_r2_std': ridge_cv.std(),
        'best_alpha': ridge_model.alpha_
    }

    # --- Lasso Regression ---
    lasso_model = LassoCV(cv=5, random_state=42)
    lasso_model.fit(X_train_scaled, y_train)
    lasso_pred = lasso_model.predict(X_test_scaled)
    lasso_cv = cross_val_score(lasso_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Lasso Regression'] = {
        'r2': r2_score(y_test, lasso_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, lasso_pred)),
        'cv_r2_mean': lasso_cv.mean(),
        'cv_r2_std': lasso_cv.std(),
        'best_alpha': lasso_model.alpha_,
        'coefficients': dict(zip(selected_features, lasso_model.coef_))
    }

    # --- Random Forest ---
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_cv = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
    results['models']['Random Forest'] = {
        'r2': r2_score(y_test, rf_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, rf_pred)),
        'cv_r2_mean': rf_cv.mean(),
        'cv_r2_std': rf_cv.std(),
        'feature_importances': dict(zip(selected_features, rf_model.feature_importances_))
    }

    # Print summary
    print(f"\n===== MODEL COMPARISON: {target} =====")
    for name, m in results['models'].items():
        print(f"{name:<25} R²: {m['r2']:.4f}  CV R²: {m['cv_r2_mean']:.4f}  RMSE: {m['rmse']:.4f}")

    return results


In [ ]:
def generate_pdf_report(all_results, filename="regression_report.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=0.75*inch, leftMargin=0.75*inch,
                            topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story = []

    # Custom styles
    title_style = ParagraphStyle('CustomTitle', parent=styles['Title'], fontSize=20, spaceAfter=10)
    heading_style = ParagraphStyle('CustomHeading', parent=styles['Heading1'], fontSize=14,
                                   textColor=colors.HexColor('#2C3E50'), spaceAfter=6)
    subheading_style = ParagraphStyle('SubHeading', parent=styles['Heading2'], fontSize=11,
                                      textColor=colors.HexColor('#2980B9'), spaceAfter=4)
    normal_style = styles['Normal']
    header_style = ParagraphStyle('TableHeader', parent=styles['Normal'], textColor=colors.white, 
                                  fontName='Helvetica-Bold', fontSize=9, alignment=1)

    # Title Page
    story.append(Spacer(1, 0.8*inch))
    story.append(Paragraph("Baseball Statcast Multi_Team", title_style))
    story.append(Paragraph("Regression Analysis Report", title_style))
    story.append(Spacer(1, 0.3*inch))
    story.append(Paragraph("Targets: TB, BB, AB, H", styles['Heading2']))
    story.append(Paragraph("Models: Linear, Ridge, Lasso, Random Forest", styles['Heading2']))
    story.append(PageBreak())

    # Summary comparison table across all targets
    story.append(Paragraph("Overall Model Performance Summary", heading_style))
    story.append(Spacer(1, 0.1*inch))

    summary_data = [[
    Paragraph('Target', header_style),
    Paragraph('Best Model', header_style),
    Paragraph('R<super>2</super>', header_style),
    Paragraph('CV R<super>2</super>', header_style),
    Paragraph('RMSE', header_style)
]]
    for res in all_results:
        if not res['models']:
            continue
        best_model = max(res['models'].items(), key=lambda x: x[1]['r2'])
        name, m = best_model
        summary_data.append([
            res['target'], name,
            f"{m['r2']:.4f}",
            f"{m['cv_r2_mean']:.4f} ± {m['cv_r2_std']:.4f}",
            f"{m['rmse']:.4f}"
        ])

    summary_table = Table(summary_data, colWidths=[0.8*inch, 1.8*inch, 0.9*inch, 1.8*inch, 0.9*inch])
    summary_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2C3E50')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 8),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#ECF0F1'), colors.white]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 6),
    ]))
    story.append(summary_table)
    story.append(PageBreak())

    # Detailed section per target
    for res in all_results:
        target = res['target']
        story.append(Paragraph(f"Target Variable: {target}", heading_style))

        # Selected features
        story.append(Paragraph("Selected Features (|correlation| > 0.5):", subheading_style))
        if res['selected_features']:
            story.append(Paragraph(", ".join(res['selected_features']), normal_style))
        else:
            story.append(Paragraph("No features met the threshold.", normal_style))
            story.append(PageBreak())
            continue

        story.append(Spacer(1, 0.15*inch))

        # Model comparison table
        story.append(Paragraph("Model Comparison", subheading_style))
        model_data = [[
            Paragraph('Model', header_style),
            Paragraph('R<super>2</super>', header_style),
            Paragraph('CV R<super>2</super>', header_style),
            Paragraph('RMSE', header_style),
            Paragraph('Best Alpha', header_style)
        ]]

        for model_name, m in res['models'].items():
            alpha = f"{m.get('best_alpha', 'N/A'):.4f}" if isinstance(m.get('best_alpha'), float) else 'N/A'
            model_data.append([
                model_name,
                f"{m['r2']:.4f}",
                f"{m['cv_r2_mean']:.4f} ± {m['cv_r2_std']:.4f}",
                f"{m['rmse']:.4f}",
                alpha
            ])

        model_table = Table(model_data, colWidths=[1.6*inch, 0.9*inch, 1.8*inch, 0.9*inch, 0.9*inch])
        model_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
            ('TOPPADDING', (0, 0), (-1, -1), 3),
        ]))
        story.append(model_table)
        story.append(Spacer(1, 0.2*inch))

        # Lasso coefficients
        if 'Lasso Regression' in res['models'] and 'coefficients' in res['models']['Lasso Regression']:
            story.append(Paragraph("Lasso Coefficients", subheading_style))
            coef_data = [['Feature', 'Coefficient']]
            sorted_coefs = sorted(res['models']['Lasso Regression']['coefficients'].items(),
                                  key=lambda x: abs(x[1]), reverse=True)
            for feat, coef in sorted_coefs:
                coef_data.append([feat, f"{coef:.4f}"])

            coef_table = Table(coef_data, colWidths=[3*inch, 1.5*inch])
            coef_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 9),
                ('ALIGN', (1, 0), (1, -1), 'CENTER'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
                ('TOPPADDING', (0, 0), (-1, -1), 3),
            ]))
            story.append(coef_table)
            story.append(Spacer(1, 0.2*inch))

        # Random Forest feature importances
        if 'Random Forest' in res['models'] and 'feature_importances' in res['models']['Random Forest']:
            story.append(Paragraph("Random Forest Feature Importances", subheading_style))
            imp_data = [['Feature', 'Importance']]
            sorted_imp = sorted(res['models']['Random Forest']['feature_importances'].items(),
                                key=lambda x: x[1], reverse=True)
            for feat, imp in sorted_imp:
                imp_data.append([feat, f"{imp:.4f}"])

            imp_table = Table(imp_data, colWidths=[3*inch, 1.5*inch])
            imp_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 9),
                ('ALIGN', (1, 0), (1, -1), 'CENTER'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
                ('TOPPADDING', (0, 0), (-1, -1), 3),
            ]))
            story.append(imp_table)

        story.append(PageBreak())

    doc.build(story)
    print(f"\nPDF report saved to: {filename}")

In [ ]:
# --- Define base features ---
base_features = [
    'Zone_%', 'Zone_Swing_%', 'Zone_Contact_%', 'Chase_%',
    'Chase_Contact_%', 'Edge_%', '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%',
    'Meatball_%', 'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%',
    'Pull_%', 'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
    'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
    'Hard_Hit_%', 'Exit_Velocity', 'Launch_Angle'
]

# --- Run all targets and collect results ---
all_results = []
for target in ['TB', 'BB', 'AB', 'H']:
    result = run_regression_analysis(target, base_features, statcast)
    all_results.append(result)

# --- Generate PDF ---
generate_pdf_report(all_results, filename="multi_team_statcast_regression_report.pdf")
os.startfile('multi_team_statcast_regression_report.pdf')


  TARGET VARIABLE: TB


KeyError: "None of [Index(['Zone_%', 'Zone_Swing_%', 'Zone_Contact_%', 'Chase_%',\n       'Chase_Contact_%', 'Edge_%', '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%',\n       'Meatball_%', 'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%',\n       'Pull_%', 'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',\n       'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',\n       'Hard_Hit_%', 'Exit_Velocity', 'Launch_Angle', 'TB'],\n      dtype='object')] are in the [columns]"